In [41]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("Python:", sys.executable)
print("Work Dir:", Path.cwd())

DATA_DIR = Path("simple_ant_sim/data/dataset_20261025_104552_105991")
OUT_DIR = Path("simple_ant_sim/data/simple_model")

Python: /home/spatel/miniconda3/envs/ant_sim/bin/python
Work Dir: /home/spatel/Desktop/Projects/collective_intelligence/ant_sim


In [42]:
print("DATA_DIR:", DATA_DIR.resolve(), "exists:", DATA_DIR.exists())

DATA_DIR: /home/spatel/Desktop/Projects/collective_intelligence/ant_sim/simple_ant_sim/data/dataset_20261025_104552_105991 exists: True


In [43]:
# find the runs
splits = {}
print("====== Runs split ======")
for split in ["train","val","test"]:
    splits[split] = sorted(p.parent for p in (DATA_DIR / split).glob("*/traj.npz"))
    print(split, len(splits[split]), "runs")

print("\n\n====== Observations =======")
data = np.load(splits["train"][0] / "traj.npz")
for k in data.files:
    print(k, data[k].shape)

====== Runs split ======
train 60 runs
val 20 runs
test 20 runs


====== Observations =======
state (1500, 100, 19)
action (1500, 100)
prev_action (1500, 100, 4)
dec_pick (1500, 100)
dec_drop (1500, 100)
delivered (1501,)
epsilon ()
seed ()


In [44]:
# Load and validate each rein
# Check if state/action dimensions match : T=1500 and N=100

def load_run(run_dir):
    data = np.load(run_dir / "traj.npz")
    T, N, _ = data["state"].shape
    return {
        "state": data["state"].reshape(-1, 19).astype(np.float32),
        "action": data["action"].reshape(-1).astype(np.int64),
        "pick": data["dec_pick"].reshape(-1).astype(np.float32),
        "drop": data["dec_drop"].reshape(-1).astype(np.float32),
        "epsilon": np.full(T * N, float(data["epsilon"]), dtype=np.float32),
        "seed": np.full(T * N, int(data["seed"]), dtype=np.int64),
    }


In [45]:
# Load and see the data.

parts = {}
for split, run_list in splits.items():
    runs = [load_run(run_dir) for run_dir in run_list]
    parts[split] = {key: np.concatenate([r[key] for r in runs]) for key in runs[0]}
    print(split, len(run_list), "runs,", len(parts[split]["action"]), "rows")

OUT_DIR.mkdir(parents=True, exist_ok=True)
for split, joined in parts.items():
    np.savez(OUT_DIR / f"{split}.npz")

train 60 runs, 9000000 rows
val 20 runs, 3000000 rows
test 20 runs, 3000000 rows


In [46]:
def normalize(s):
    s = s.astype(np.float32).copy()
    s[:, 3:11] /= 20.0
    for a, b in [(11, 15), (15, 19)]:
        block = s[:, a:b]
        lo, hi = block.min(1, keepdims=True), block.max(1, keepdims=True)
        s[:, a:b] = np.where(hi > lo, (block - lo) / np.maximum(hi - lo, 1e-8), 0.0)
    return s

In [47]:
## Simple ant database
class AntDataset(Dataset):
    def __init__(self, d, norm=True):
        state = normalize(d["state"]) if norm else d["state"]
        self.state = torch.tensor(d["state"])
        self.action = torch.tensor(d["action"])
        self.pick = torch.tensor(d["pick"])
        self.drop = torch.tensor(d["drop"])
        self.epsilon = torch.tensor(d["epsilon"])

    def __len__(self):
        return len(self.action)

    def __getitem__(self, i):
        return {
            "state" : self.state[i],
            "action": self.action[i],
            "pick": self.pick[i],
            "drop": self.drop[i],
            "epsilon": self.epsilon[i]
        }


In [48]:
train_loader = DataLoader(AntDataset(parts["train"]), batch_size = 256, shuffle=True)
val_loader = DataLoader(AntDataset(parts["val"]), batch_size = 256)
test_loader = DataLoader(AntDataset(parts["test"]), batch_size = 256)

In [49]:
# Batch Test

print("===== Single Object in Datatset ======")
batch = next(iter(train_loader))
for k, v in batch.items():
    print(k, tuple(v.shape), v.dtype)

print("\n===== Dataset Stats ======")

tr = parts["train"]
print("action counts:", np.bincount(tr["action"]))
print("pick rate:", tr["pick"].mean())
print("drop rate:", tr["drop"].mean())
print("epsilons:", np.unique(tr["epsilon"]))

===== Single Object in Datatset ======
state (256, 19) torch.float32
action (256,) torch.int64
pick (256,) torch.float32
drop (256,) torch.float32
epsilon (256,) torch.float32

===== Dataset Stats ======
action counts: [2229001 2220072 2285763 2265164]
pick rate: 0.004228222
drop rate: 0.0039824443
epsilons: [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]


In [50]:
# Model Loading
device = "cuda" if torch.cuda.is_available() else "cpu"

In [51]:
class MLP(nn.Module):
    def __init__(self, i_dim=19, n_dim=128):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(i_dim, n_dim),
            nn.ReLU(),
            nn.Linear(n_dim, n_dim),
            nn.ReLU()
        )
        self.action_head = nn.Linear(n_dim, 4)
        self.pick_head = nn.Linear(n_dim, 1)
        self.drop_head = nn.Linear(n_dim, 1)

    def forward(self, x):
        z = self.ffn(x)
        return self.action_head(z), self.pick_head(z).squeeze(-1), self.drop_head(z).squeeze(-1)

In [52]:
model = MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# print(model)
# print(sum(p.numel() for p in model.parameters()))

In [53]:
ce = nn.CrossEntropyLoss()
p_rate = float(tr["pick"].mean())
d_rate = float(tr["drop"].mean())
print("pick rate:", p_rate, "drop rate:", d_rate)

bce_pick = nn.BCEWithLogitsLoss()
bce_drop = nn.BCEWithLogitsLoss()
# bce_pick = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - p_rate) / p_rate, device=device))
# bce_drop = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - d_rate) / d_rate, device=device))

pick rate: 0.00422822218388319 drop rate: 0.003982444293797016


In [54]:
# print("pick pos_weight:", bce_pick.pos_weight.item(), " drop pos_weight:", bce_drop.pos_weight.item())

In [55]:
def true_ceiling(s, eps):
    carrying = s[:, 2] > 0.5
    target = np.where(carrying[:, None], s[:, 7:11], s[:, 3:7])
    smell = np.where(carrying[:, None], s[:, 15:19], s[:, 11:15])
    signal = np.where(target.max(1, keepdims=True) > 0, target, smell)  # same priority as the brain
    k = (signal == signal.max(1, keepdims=True)).sum(1)                # 4 when there is no signal
    return (1 - eps) / k + eps / 4, k


def loss_floor(s, eps):
    _, k = true_ceiling(s, eps)
    p_best = (1 - eps) / k + eps / 4    # each tied best direction
    p_other = eps / 4                   # each remaining direction
    H = -(k * p_best * np.log(p_best) + (4 - k) * p_other * np.log(p_other))
    return H.mean()
    
@torch.no_grad()
def val_check(loader):
    model.eval()
    s, eps, act, pred = [], [], [], []
    ce_sum = 0.0
    for b in loader:
        a_logit, _, _ = model(b["state"].to(device))
        ce_sum += nn.functional.cross_entropy(
            a_logit, b["action"].long().to(device), reduction="sum"
        ).item()
        s.append(b["state"])
        eps.append(b["epsilon"])
        act.append(b["action"])
        pred.append(a_logit.argmax(-1).cpu())
    model.train()

    s, eps, act, pred = (torch.cat(x).numpy() for x in (s, eps, act, pred))
    _, k = true_ceiling(s, eps)
    m = np.isclose(eps, 0.1) & (k == 1)
    acc_k1 = (pred[m] == act[m]).mean()
    gap = ce_sum / len(act) - loss_floor(s, eps)
    return acc_k1, gap
    
# Model Training
def train_step(epochs, log_every=500):
    model.train()
    print("steps per epoch:", len(train_loader))

    step = 0
    for epoch in range(1, epochs + 1):
        running, n = 0.0, 0
        for b in train_loader:
            a_logit, p_logit, d_logit = model(b["state"].to(device))
            loss = (ce(a_logit, b["action"].long().to(device))
                    + bce_pick(p_logit, b["pick"].float().to(device))
                    + bce_drop(d_logit, b["drop"].float().to(device))
                   )
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            step += 1
            running += loss.item()
            n += 1
            if step % log_every == 0:
                print("Training: ", step, ", Avg loss", round(running / n, 4))
                running, n = 0.0, 0

        acc_k1, gap = val_check(val_loader)
        print(f"== epoch {epoch} done: val k=1 acc at eps 0.1 = {acc_k1:.3f} (target 0.925), "
              f"action loss above floor = {gap:.4f}")

In [57]:
train_step(2)

steps per epoch: 35157
Training:  500 , Avg loss 1.4723
Training:  1000 , Avg loss 1.3626
Training:  1500 , Avg loss 1.3372
Training:  2000 , Avg loss 1.3229
Training:  2500 , Avg loss 1.3125
Training:  3000 , Avg loss 1.3036
Training:  3500 , Avg loss 1.295
Training:  4000 , Avg loss 1.2903
Training:  4500 , Avg loss 1.2827
Training:  5000 , Avg loss 1.2767
Training:  5500 , Avg loss 1.2748
Training:  6000 , Avg loss 1.2699
Training:  6500 , Avg loss 1.267
Training:  7000 , Avg loss 1.262
Training:  7500 , Avg loss 1.259
Training:  8000 , Avg loss 1.2577
Training:  8500 , Avg loss 1.2538
Training:  9000 , Avg loss 1.2511
Training:  9500 , Avg loss 1.2495
Training:  10000 , Avg loss 1.2476
Training:  10500 , Avg loss 1.2459
Training:  11000 , Avg loss 1.2458
Training:  11500 , Avg loss 1.243
Training:  12000 , Avg loss 1.2431
Training:  12500 , Avg loss 1.2402
Training:  13000 , Avg loss 1.237
Training:  13500 , Avg loss 1.2362
Training:  14000 , Avg loss 1.2369
Training:  14500 , Avg 

In [58]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    s, eps, a_true, a_pred, p_true, p_pred, d_true, d_pred = ([] for _ in range(8))
    for b in loader:
        a_logit, p_logit, d_logit = model(b["state"].to(device))
        s.append(b["state"])
        eps.append(b["epsilon"])
        a_true.append(b["action"])
        a_pred.append(a_logit.argmax(-1).cpu())
        p_true.append(b["pick"])
        p_pred.append((p_logit > 0).float().cpu())
        d_true.append(b["drop"])
        d_pred.append((d_logit > 0).float().cpu())

    cat = lambda xs: torch.cat(xs).numpy()
    s, eps, a_true, a_pred = cat(s), cat(eps), cat(a_true), cat(a_pred)
    ceiling, k = true_ceiling(s, eps)

    print("eps   acc    ceiling")
    for e in np.unique(eps):
        m = np.isclose(eps, e)
        print(f"{e:.1f}   {(a_true[m] == a_pred[m]).mean():.3f}  {ceiling[m].mean():.3f}")
    print("rows by tied directions (k = 1, 2, 3, 4):", np.bincount(k, minlength=5)[1:])

    for name, t, p in [("pick", cat(p_true), cat(p_pred)), ("drop", cat(d_true), cat(d_pred))]:
        tp = ((t == 1) & (p == 1)).sum()
        print(f"{name}: precision {tp / max(p.sum(), 1):.3f}  recall {tp / max(t.sum(), 1):.3f}")

    return s, eps, a_true, a_pred

In [59]:
Val_s, Val_eps, Val_act, Val_pred = evaluate(val_loader)

eps   acc    ceiling
0.1   0.621  0.662
0.2   0.570  0.603
0.3   0.530  0.556
0.4   0.499  0.525
0.5   0.443  0.462
0.6   0.415  0.434
0.7   0.361  0.376
0.8   0.310  0.323
0.9   0.273  0.284
1.0   0.250  0.250
rows by tied directions (k = 1, 2, 3, 4): [1544657   47545    2693 1405105]
pick: precision 1.000  recall 1.000
drop: precision 1.000  recall 1.000


In [60]:
test_s, test_eps, test_act, test_pred = evaluate(test_loader)

eps   acc    ceiling
0.1   0.646  0.692
0.2   0.594  0.646
0.3   0.539  0.607
0.4   0.526  0.558
0.5   0.486  0.512
0.6   0.442  0.463
0.7   0.379  0.395
0.8   0.339  0.351
0.9   0.286  0.294
1.0   0.249  0.250
rows by tied directions (k = 1, 2, 3, 4): [1859961   74504    4892 1060643]
pick: precision 1.000  recall 1.000
drop: precision 0.998  recall 1.000


In [61]:
train_s, train_eps, train_act, train_pred = evaluate(train_loader)

eps   acc    ceiling
0.1   0.662  0.689
0.2   0.621  0.647
0.3   0.575  0.599
0.4   0.519  0.538
0.5   0.474  0.492
0.6   0.418  0.432
0.7   0.374  0.386
0.8   0.325  0.337
0.9   0.277  0.284
1.0   0.251  0.250
rows by tied directions (k = 1, 2, 3, 4): [5083350  164172   12232 3740246]
pick: precision 1.000  recall 1.000
drop: precision 1.000  recall 1.000


In [62]:
splits = {
    "train": (train_s, train_eps, train_act, train_pred),
    "val": (Val_s, Val_eps, Val_act, Val_pred),
    "test": (test_s, test_eps, test_act, test_pred),
}

def best_dir(s):
    carrying = s[:, 2] > 0.5
    target = np.where(carrying[:, None], s[:, 7:11], s[:, 3:7])
    smell = np.where(carrying[:, None], s[:, 15:19], s[:, 11:15])
    signal = np.where(target.max(1, keepdims=True) > 0, target, smell)
    return signal.argmax(1)

rows = []
for name, (s, eps, act, pred) in splits.items():
    ceiling, k = true_ceiling(s, eps)
    best = best_dir(s)
    for e in np.unique(eps):
        m = np.isclose(eps, e)
        m1 = m & (k == 1)
        rows.append(dict(
            split=name,
            eps=round(float(e), 1),
            acc=(pred[m] == act[m]).mean(),
            ceiling=ceiling[m].mean(),
            rule_k1=(act[m1] == best[m1]).mean(),    # does the data follow argmax?
            model_k1=(pred[m1] == act[m1]).mean(),   # does the model?
            expected_k1=(1 - e) + e / 4,
        ))

df = pd.DataFrame(rows)
order = ["train", "val", "test"]

print("Accuracy vs ceiling (all rows)")
acc = df.pivot(index="eps", columns="split", values="acc")[order]
acc["ceiling"] = df[df.split == "val"].set_index("eps")["ceiling"]
print(acc.round(3))

def loss_floor(s, eps):
    _, k = true_ceiling(s, eps)
    p_best = (1 - eps) / k + eps / 4    # each tied best direction
    p_other = eps / 4                   # each remaining direction
    H = -(k * p_best * np.log(p_best) + (4 - k) * p_other * np.log(p_other))
    return H.mean()

print("loss floor (train):", loss_floor(train_s, train_eps))

print("\nk = 1 rows only")
k1 = df.pivot(index="eps", columns="split", values="model_k1")[order].add_prefix("model_")
k1["data_rule"] = df[df.split == "val"].set_index("eps")["rule_k1"]
k1["expected"] = df[df.split == "val"].set_index("eps")["expected_k1"]
print(k1.round(3))

Accuracy vs ceiling (all rows)
split  train    val   test  ceiling
eps                                
0.1    0.662  0.621  0.646    0.662
0.2    0.621  0.570  0.594    0.603
0.3    0.575  0.530  0.539    0.556
0.4    0.519  0.499  0.526    0.525
0.5    0.474  0.443  0.486    0.462
0.6    0.418  0.415  0.442    0.434
0.7    0.374  0.361  0.379    0.376
0.8    0.325  0.310  0.339    0.323
0.9    0.277  0.273  0.286    0.284
1.0    0.251  0.250  0.249    0.250
loss floor (train): 1.1524001390030738

k = 1 rows only
split  model_train  model_val  model_test  data_rule  expected
eps                                                           
0.1          0.881      0.858       0.855      0.925     0.925
0.2          0.809      0.794       0.767      0.850     0.850
0.3          0.738      0.729       0.674      0.775     0.775
0.4          0.669      0.656       0.653      0.701     0.700
0.5          0.596      0.591       0.587      0.627     0.625
0.6          0.525      0.519       0.51

In [63]:
print("steps per epoch:", len(train_loader))

steps per epoch: 35157


In [64]:
for name, (s, eps, act, pred) in splits.items():
    target = np.where((s[:, 2] > 0.5)[:, None], s[:, 7:11], s[:, 3:7])
    _, k = true_ceiling(s, eps)
    best = best_dir(s)
    by_see = target.max(1) > 0
    m = k == 1
    print(f"{name}: agrees with rule  see-driven {(pred[m & by_see] == best[m & by_see]).mean():.3f}  "
          f"smell-driven {(pred[m & ~by_see] == best[m & ~by_see]).mean():.3f}")

train: agrees with rule  see-driven 0.980  smell-driven 0.916
val: agrees with rule  see-driven 0.975  smell-driven 0.893
test: agrees with rule  see-driven 0.985  smell-driven 0.883


In [65]:
bins = [0, 0.01, 0.03, 0.1, 0.3, 1.01]

for name, (s, eps, act, pred) in splits.items():
    carrying = s[:, 2] > 0.5
    target = np.where(carrying[:, None], s[:, 7:11], s[:, 3:7])
    smell = np.where(carrying[:, None], s[:, 15:19], s[:, 11:15])
    _, k = true_ceiling(s, eps)
    best = best_dir(s)

    m = (k == 1) & (target.max(1) == 0)
    top2 = np.sort(smell[m], axis=1)[:, -2:]
    margin = top2[:, 1] - top2[:, 0]
    ok = pred[m] == best[m]
    idx = np.digitize(margin, bins) - 1

    print(name)
    for i in range(len(bins) - 1):
        sel = idx == i
        if sel.any():
            print(f"  margin {bins[i]:.2f} to {bins[i + 1]:.2f}: rows {sel.sum():>8}  agree {ok[sel].mean():.3f}")            

train
  margin 0.00 to 0.01: rows   756406  agree 0.728
  margin 0.01 to 0.03: rows  1864542  agree 0.931
  margin 0.03 to 0.10: rows  1802555  agree 0.971
  margin 0.10 to 0.30: rows   301600  agree 0.958
  margin 0.30 to 1.01: rows    86337  agree 0.949
val
  margin 0.00 to 0.01: rows   202754  agree 0.626
  margin 0.01 to 0.03: rows   652429  agree 0.912
  margin 0.03 to 0.10: rows   505030  agree 0.960
  margin 0.10 to 0.30: rows    97903  agree 0.962
  margin 0.30 to 1.01: rows    24531  agree 0.925
test
  margin 0.00 to 0.01: rows   305189  agree 0.655
  margin 0.01 to 0.03: rows   569548  agree 0.889
  margin 0.03 to 0.10: rows   681345  agree 0.958
  margin 0.10 to 0.30: rows   126638  agree 0.967
  margin 0.30 to 1.01: rows    55000  agree 0.968


In [ ]:
# --------

In [67]:
# New Mode 

import numpy as np
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"


# ---------- helpers (defined here so a restart can't lose them) ----------
def true_ceiling(s, eps):
    carrying = s[:, 2] > 0.5
    target = np.where(carrying[:, None], s[:, 7:11], s[:, 3:7])
    smell = np.where(carrying[:, None], s[:, 15:19], s[:, 11:15])
    signal = np.where(target.max(1, keepdims=True) > 0, target, smell)
    k = (signal == signal.max(1, keepdims=True)).sum(1)
    return (1 - eps) / k + eps / 4, k


def best_dir(s):
    carrying = s[:, 2] > 0.5
    target = np.where(carrying[:, None], s[:, 7:11], s[:, 3:7])
    smell = np.where(carrying[:, None], s[:, 15:19], s[:, 11:15])
    signal = np.where(target.max(1, keepdims=True) > 0, target, smell)
    return signal.argmax(1)


def loss_floor(s, eps):
    _, k = true_ceiling(s, eps)
    p_best = (1 - eps) / k + eps / 4
    p_other = eps / 4
    H = -(k * p_best * np.log(p_best) + (4 - k) * p_other * np.log(p_other))
    return H.mean()


# ---------- model: g = the pure brain ----------
class DirScorer(nn.Module):
    def __init__(self, h=64):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(7, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))
        self.pick_head = nn.Linear(3, 1)
        self.drop_head = nn.Linear(3, 1)

    def forward(self, x):
        flags = x[:, :3]
        per_dir = torch.stack([x[:, 3:7], x[:, 7:11], x[:, 11:15], x[:, 15:19]], dim=-1)  # (B, 4, 4)
        tokens = torch.cat([per_dir, flags[:, None, :].expand(-1, 4, -1)], dim=-1)        # (B, 4, 7)
        a_logit = self.score(tokens).squeeze(-1)                                          # (B, 4)
        return a_logit, self.pick_head(flags).squeeze(-1), self.drop_head(flags).squeeze(-1)


# ---------- loss: brain with prob 1 - eps, random with prob eps ----------
def mixture_nll(a_logit, a, eps, reduction="mean"):
    p_brain = a_logit.softmax(-1).gather(1, a[:, None]).squeeze(1)
    p = (1 - eps) * p_brain + eps / 4
    nll = -torch.log(p + 1e-12)
    return nll.mean() if reduction == "mean" else nll.sum()


model = DirScorer().to(device)          # or MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
bce_pick = nn.BCEWithLogitsLoss()
bce_drop = nn.BCEWithLogitsLoss()


# ---------- validation ----------
@torch.no_grad()
def val_check(loader):
    model.eval()
    s, eps, pred, top = [], [], [], []
    nll_sum, n = 0.0, 0
    for b in loader:
        a_logit, _, _ = model(b["state"].to(device))
        nll_sum += mixture_nll(a_logit, b["action"].long().to(device),
                               b["epsilon"].float().to(device), reduction="sum").item()
        prob = a_logit.softmax(-1)
        s.append(b["state"]); eps.append(b["epsilon"])
        pred.append(prob.argmax(-1).cpu()); top.append(prob.max(-1).values.cpu())
        n += len(a_logit)
    model.train()

    s, eps, pred, top = (torch.cat(x).numpy() for x in (s, eps, pred, top))
    _, k = true_ceiling(s, eps)
    m1 = k == 1
    agree = (pred[m1] == best_dir(s)[m1]).mean()     # target 1.0
    sharp = top[m1].mean()                           # how confident g is on clear rows
    gap = nll_sum / n - loss_floor(s, eps)           # target 0
    return agree, sharp, gap


# ---------- training ----------
def train(epochs, log_every=500):
    model.train()
    print("steps per epoch:", len(train_loader))
    step = 0
    for epoch in range(1, epochs + 1):
        running, n = 0.0, 0
        for b in train_loader:
            a_logit, p_logit, d_logit = model(b["state"].to(device))
            loss = (mixture_nll(a_logit, b["action"].long().to(device), b["epsilon"].float().to(device))
                    + bce_pick(p_logit, b["pick"].float().to(device))
                    + bce_drop(d_logit, b["drop"].float().to(device)))
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            step += 1
            running += loss.item()
            n += 1
            if step % log_every == 0:
                print(f"epoch {epoch}  step {step}  avg loss {running / n:.4f}")
                running, n = 0.0, 0

        agree, sharp, gap = val_check(val_loader)
        print(f"== epoch {epoch}: agrees with rule {agree:.3f}  sharpness {sharp:.3f}  "
              f"loss above floor {gap:.4f}")


train(2)

steps per epoch: 35157
epoch 1  step 500  avg loss 2.3675
epoch 1  step 1000  avg loss 1.8884
epoch 1  step 1500  avg loss 1.6567
epoch 1  step 2000  avg loss 1.5281
epoch 1  step 2500  avg loss 1.4408
epoch 1  step 3000  avg loss 1.3814
epoch 1  step 3500  avg loss 1.3383
epoch 1  step 4000  avg loss 1.3053
epoch 1  step 4500  avg loss 1.2810
epoch 1  step 5000  avg loss 1.2676
epoch 1  step 5500  avg loss 1.2506
epoch 1  step 6000  avg loss 1.2379
epoch 1  step 6500  avg loss 1.2300
epoch 1  step 7000  avg loss 1.2224
epoch 1  step 7500  avg loss 1.2131
epoch 1  step 8000  avg loss 1.2101
epoch 1  step 8500  avg loss 1.2013
epoch 1  step 9000  avg loss 1.1957
epoch 1  step 9500  avg loss 1.1947
epoch 1  step 10000  avg loss 1.1906
epoch 1  step 10500  avg loss 1.1875
epoch 1  step 11000  avg loss 1.1855
epoch 1  step 11500  avg loss 1.1853
epoch 1  step 12000  avg loss 1.1851
epoch 1  step 12500  avg loss 1.1804
epoch 1  step 13000  avg loss 1.1809
epoch 1  step 13500  avg loss 1.178

In [73]:
import sys, inspect
from pathlib import Path

try:
    from simple_ant_sim.simulation import Simulation
    from simple_ant_sim.config import Config
except ImportError:
    sys.path.insert(0, str(Path.cwd() / "simple_ant_sim"))
    from simulation import Simulation
    from config import Config

sim_mod = sys.modules[Simulation.__module__]
print("Simulation from:", sim_mod.__file__)
print("decide in that module:", getattr(sim_mod, "decide", "NOT FOUND"))
print("Simulation(...) takes:", inspect.signature(Simulation.__init__))
print("Config(...) takes:   ", inspect.signature(Config))

Simulation from: /home/spatel/Desktop/Projects/collective_intelligence/ant_sim/simple_ant_sim/simulation.py
decide in that module: <function decide at 0x7ebe25979260>
Simulation(...) takes: (self, cfg)
Config(...) takes:    (width: int = 100, height: int = 100, nest: tuple = (50, 50), nest_radius: int = 3, n_ants: int = 100, speed: float = 1.0, sense_radius: int = 3, deposit: float = 1.0, deposit_decay: float = 0.01, evaporation: float = 0.001, min_phe: float = 0.01, epsilon: float = 0.25, keep_direction: float = 0.75, seed: int = 0) -> None


In [77]:
from pathlib import Path

files = sorted(Path.cwd().rglob("*.npz"))
print(len(files), "npz files")
for f in files[:5]:
    with np.load(f) as r:
        print(f.relative_to(Path.cwd()), r.files)
        
S, E, A, P = [], [], [], []
for f in files:
    with np.load(f) as r:
        if not {"state", "action", "epsilon", "prev_action"} <= set(r.files):
            continue
        S.append(r["state"].reshape(-1, 19))
        A.append(r["action"].reshape(-1))
        P.append(r["prev_action"].reshape(-1, 4))
        E.append(np.full(r["action"].size, float(r["epsilon"])))
    if len(S) == 20:
        break

print(len(S), "runs loaded")
d = dict(state=np.concatenate(S), action=np.concatenate(A),
         prev_action=np.concatenate(P), epsilon=np.concatenate(E))

131 npz files
simple_ant_sim/data/brain_simple/test.npz []
simple_ant_sim/data/brain_simple/train.npz []
simple_ant_sim/data/brain_simple/val.npz []
simple_ant_sim/data/dataset_20261025_104552_105991/test/seed008_eps0.1_p2_a100/traj.npz ['state', 'action', 'prev_action', 'dec_pick', 'dec_drop', 'delivered', 'epsilon', 'seed']
simple_ant_sim/data/dataset_20261025_104552_105991/test/seed008_eps0.2_p2_a100/traj.npz ['state', 'action', 'prev_action', 'dec_pick', 'dec_drop', 'delivered', 'epsilon', 'seed']
20 runs loaded


In [86]:
import sys, types, copy, time
from collections import defaultdict
import numpy as np
import pandas as pd
import torch

torch.set_num_threads(1)                      # tiny one-row model calls are much faster single-threaded

sim_mod = sys.modules[Simulation.__module__]  # the module where step_ant looks up `decide`
original_decide = sim_mod.decide
assert original_decide.__module__.endswith("brain"), f"unexpected decide: {original_decide}"

cpu_model = copy.deepcopy(model).cpu().eval()
STATS = defaultdict(float)


# ---------- brains ----------
@torch.no_grad()
def learned_decide(ant, local, rng, cfg):
    x = torch.from_numpy(normalize(ant.obs[None]))
    a_logit, p_logit, d_logit = cpu_model(x)
    prob = a_logit.softmax(-1)[0].numpy().astype(np.float64)
    STATS["top_prob_sum"] += prob.max()

    if rng.random() < cfg.epsilon:
        d = rng.integers(4)
        STATS["explore"] += 1
    else:
        d = rng.choice(4, p=prob / prob.sum())

    action = np.zeros(4, dtype=np.float32)
    action[d] = 1.0
    return types.SimpleNamespace(
        action=action,
        pick=bool(p_logit.item() > 0),
        drop=bool(d_logit.item() > 0),
        mode="learned",
    )


def counting(brain, name):
    def wrapped(ant, local, rng, cfg):
        t = time.perf_counter()
        dec = brain(ant, local, rng, cfg)
        STATS["time"] += time.perf_counter() - t
        STATS["calls"] += 1
        STATS["pick"] += bool(dec.pick)
        STATS["drop"] += bool(dec.drop)
        STATS[f"dir{int(np.argmax(dec.action))}"] += 1

        at_home, carrying = ant.obs[1] > 0.5, ant.obs[2] > 0.5
        STATS["carry_steps"] += carrying
        if at_home and carrying:
            STATS["home_carry"] += 1
            STATS["home_carry_drop"] += bool(dec.drop)
        return dec
    wrapped.__name__ = name
    return wrapped


BRAINS = {
    "original": counting(original_decide, "original"),
    "learned": counting(learned_decide, "learned"),
}


# ---------- running ----------
RANDOM_FOOD = False     # match whatever your training runs used


def make_sim(eps, seed):
    cfg = Config(epsilon=eps, seed=seed)
    sim = Simulation(cfg)

    if not RANDOM_FOOD:
        sim.world.add_food(80, 80, radius=4, amount_per_cell=5)
        sim.world.add_food(20, 75, radius=4, amount_per_cell=5)
        return sim

    # same random placement as run.py
    rng = np.random.default_rng(10_000 + cfg.seed)
    nx, ny = cfg.nest
    placed = 0
    while placed < 2:
        x, y = rng.integers(10, cfg.width - 10), rng.integers(10, cfg.width - 10)
        if np.hypot(x - nx, y - ny) > 20:
            sim.world.add_food(int(x), int(y), radius=4, amount_per_cell=20)
            placed += 1
    return sim

def summarize(label, total_time):
    n = max(STATS["calls"], 1)
    dirs = [STATS[f"dir{i}"] / n for i in range(4)]
    line = (f"    {label}: {int(STATS['calls'])} decisions, {1000 * STATS['time'] / n:.3f} ms each, "
            f"brain = {100 * STATS['time'] / max(total_time, 1e-9):.0f}% of run time | "
            f"picks {int(STATS['pick'])}  drops {int(STATS['drop'])} | "
            f"dir split {np.round(dirs, 3)}")
    if label == "learned":
        line += (f" | explored {STATS['explore'] / n:.3f}"
                 f"  mean top prob {STATS['top_prob_sum'] / n:.3f}"
                 f" | carrying steps {int(STATS['carry_steps'])}"
                 f"  at home while carrying {int(STATS['home_carry'])}"
                 f"  dropped there {int(STATS['home_carry_drop'])}"
                )

    print(line, flush=True)


def run(eps, seed, label, T=2000, log_every=500):
    STATS.clear()
    sim_mod.decide = BRAINS[label]
    t0 = time.time()
    try:
        sim = make_sim(eps, seed)
        delivered = [sim.world.food_delivered]
        for step in range(1, T + 1):
            sim.step()
            delivered.append(sim.world.food_delivered)
            if step % log_every == 0 or step == T:
                el = time.time() - t0
                print(f"  [{label:8s} eps {eps} seed {seed}] step {step}/{T}  "
                      f"delivered {delivered[-1] - delivered[0]}  "
                      f"{step / el:.1f} steps/s  ({el:.0f} s)", flush=True)
    finally:
        sim_mod.decide = original_decide      # always restore the real brain
    summarize(label, time.time() - t0)
    return np.array(delivered)


# ---------- 1. timing check ----------
print("== timing check (50 steps each)")
for label in BRAINS:
    t = time.time()
    run(0.1, 0, label, T=50, log_every=50)
    per_step = (time.time() - t) / 50
    print(f"  {label}: {per_step:.3f} s per step -> about {per_step * 2000 / 60:.1f} min per 2000-step run\n")


# ---------- 2. comparison ----------
QUICK = True                                   # set False for the full run
T = 500 if QUICK else 2000
SEEDS = [0] if QUICK else [0, 1, 2]
EPS_LIST = [0.1, 0.3, 0.5]

rows = []
for eps in EPS_LIST:
    for seed in SEEDS:
        print(f"\n== eps {eps}  seed {seed}")
        o = run(eps, seed, "original", T=T)
        l = run(eps, seed, "learned", T=T)
        rows.append(dict(eps=eps, seed=seed,
                         original=o[-1] - o[0], learned=l[-1] - l[0]))

res = pd.DataFrame(rows)
print("\n== food delivered over", T, "steps")
print(res.groupby("eps")[["original", "learned"]].agg(["mean", "std"]))
print("\nreal brain restored:", sim_mod.decide is original_decide)

== timing check (50 steps each)
  [original eps 0.1 seed 0] step 50/50  delivered 0  273.6 steps/s  (0 s)
    original: 5000 decisions, 0.009 ms each, brain = 25% of run time | picks 0  drops 0 | dir split [0.251 0.243 0.26  0.246]
  original: 0.004 s per step -> about 0.1 min per 2000-step run

  [learned  eps 0.1 seed 0] step 50/50  delivered 0  52.5 steps/s  (1 s)
    learned: 5000 decisions, 0.154 ms each, brain = 81% of run time | picks 0  drops 0 | dir split [0.262 0.265 0.237 0.236] | explored 0.107  mean top prob 0.486 | carrying steps 0  at home while carrying 0  dropped there 0
  learned: 0.019 s per step -> about 0.6 min per 2000-step run


== eps 0.1  seed 0
  [original eps 0.1 seed 0] step 500/500  delivered 45  262.5 steps/s  (2 s)
    original: 50000 decisions, 0.011 ms each, brain = 28% of run time | picks 64  drops 45 | dir split [0.25  0.245 0.257 0.248]
  [learned  eps 0.1 seed 0] step 500/500  delivered 4  52.6 steps/s  (10 s)
    learned: 50000 decisions, 0.153 ms 

In [87]:
import inspect
print(inspect.getsource(original_decide))

def decide(ant, local, rng, cfg):
    carrying = ant.has_food()

    target = local.see_home if carrying else local.see_food
    smell = local.smell_home if carrying else local.smell_food

    action, mode = choose_direction(target, smell, rng, cfg.epsilon)

    return Decision(
        action=action,
        pick=not carrying and local.on_food,
        drop=carrying and local.at_home,
        mode=mode,
    )

